# Formulary Lag Detector — Data Exploration

This project measures the time gap between FDA generic drug
approval and first appearance in NADAC pharmacy pricing data
(the formulary lag). Faster generic adoption means lower costs
for PBMs and members.

Data strategy: two openFDA endpoints give us everything needed.
- `drug/drugsfda` → FDA approval date per ANDA application
- `drug/ndc` → marketing_start_date (when pharmacies began selling)
- Lag = marketing_start_date minus approval_date

In [5]:
import requests
import json
from datetime import datetime

In [6]:
url = "https://api.fda.gov/drug/drugsfda.json"
params = {
    "search": "application_number:ANDA*",
    "limit": 5
}
response = requests.get(url, params=params, timeout=30)
data = response.json()

for result in data['results']:
    app_number = result.get('application_number')
    drug_name = result['products'][0]['active_ingredients'][0]['name'] if result.get('products') else 'N/A'
    orig_submission = next(
        (s for s in result.get('submissions', [])
         if s['submission_type'] == 'ORIG' and s['submission_status'] == 'AP'),
        None
    )
    approval_date = orig_submission['submission_status_date'] if orig_submission else 'N/A'
    print(f"{app_number} | {drug_name} | approved: {approval_date}")

ANDA076367 | AMCINONIDE | approved: 20030319
ANDA076648 | NITROFURANTOIN | approved: 20040322
ANDA076919 | LEVETIRACETAM | approved: 20081104
ANDA077221 | LAMIVUDINE | approved: 20170303
ANDA077527 | DROSPIRENONE | approved: 20080509


In [7]:
url = "https://api.fda.gov/drug/ndc.json"
params = {
    "search": "application_number:ANDA* AND finished:true",
    "limit": 3
}
response = requests.get(url, params=params, timeout=30)
data = response.json()

for r in data['results']:
    print(f"{r.get('generic_name')} | "
          f"app: {r.get('application_number')} | "
          f"marketing_start: {r.get('marketing_start_date')}")

Celecoxib | app: ANDA208856 | marketing_start: 20251215
triamcinolone acetonide | app: ANDA208848 | marketing_start: 20240316
Moexipril Hydrochloride | app: ANDA076204 | marketing_start: 20030508


## End-to-end lag calculation

Joining the two endpoints on application_number gives us a clean,
verifiable lag per drug. ATORVASTATIN CALCIUM is used as the
validation case since it has a well-known approval history.

In [8]:
# FDA approval date
url = "https://api.fda.gov/drug/drugsfda.json"
params = {"search": "application_number:ANDA090548", "limit": 1}
response = requests.get(url, params=params, timeout=30)
result = response.json()['results'][0]

orig = next(
    (s for s in result.get('submissions', [])
     if s['submission_type'] == 'ORIG' and s['submission_status'] == 'AP'),
    None
)
approval_date = orig['submission_status_date'] if orig else None

# Marketing start date
url2 = "https://api.fda.gov/drug/ndc.json"
params2 = {"search": "application_number:ANDA090548", "limit": 1}
ndc_result = requests.get(url2, params=params2, timeout=30).json()['results'][0]
marketing_start = ndc_result.get('marketing_start_date')

# Lag
approval_dt = datetime.strptime(approval_date, '%Y%m%d')
marketing_dt = datetime.strptime(marketing_start, '%Y%m%d')
lag_days = (marketing_dt - approval_dt).days
print(f"Drug: {result['products'][0]['active_ingredients'][0]['name']}")
print(f"FDA approved: {approval_date}")
print(f"Marketing start: {marketing_start}")
print(f"Lag: {lag_days} days ({lag_days/7:.1f} weeks)")

Drug: ATORVASTATIN CALCIUM
FDA approved: 20120529
Marketing start: 20140311
Lag: 651 days (93.0 weeks)
